# LLM-judge cục bộ trên Kaggle (llama.cpp)

Chấm đúng/sai câu trả lời của các nhánh bằng một LLM mở chạy ngay trên GPU Kaggle, thay cho Gemini API (gói miễn phí
chỉ 20 lượt/ngày). Judge chỉ đọc chữ: câu hỏi, đáp án chuẩn, câu trả lời; không xem ảnh. Prompt và cache dùng chung với
`src/eval/judge.py` (phiên bản prompt `j1`).

**Hai judge, khai báo trước (scope mục 6.1, 18/11/2026):**
- Judge chính: **Qwen3.8-27B**, GGUF `UD-Q4_K_M` (Unsloth, 16,6 GB), chia layer qua 2 T4.
- Judge đối chứng: **Gemma 4 12B-it**, GGUF QAT `q4_0` của Google (7,0 GB), 1 T4. Khác họ với model đang được chấm
  (Qwen2.5-VL), dùng để kiểm judge Qwen có thiên vị không.

**Quy tắc chọn (ghi trước khi chạy):** judge chính là Qwen3.8-27B nếu chấm nhanh hơn hoặc bằng 2 giây/câu trên 50 câu
mẫu; chậm hơn thì Gemma 4 12B làm judge chính. Judge được chọn phải đạt Cohen's κ ≥ 0,6 với nhãn người; không đạt thì
chỉ số chính quay về EM.

**Các bước:** (1) dựng llama.cpp → (2) Gemma chấm 50 câu mẫu → (3) Qwen chấm 50 câu mẫu → (4) tính κ, áp quy tắc →
(5) judge được chọn chấm toàn bộ kết quả dev trong gói → (6) đóng gói kết quả.

**Cần:** dataset `da2-judge` (file `data/processed/kaggle/da2-judge.zip`, tạo bằng
`python -X utf8 -m src.eval.judge pack`), GPU **T4 x2**, Internet **On**.
Lần đầu mất khoảng 45–70 phút (dựng llama.cpp 15–25 phút, tải 24 GB model, chấm).

**Kết quả:** tải `/kaggle/working/judge_results.zip`, giải nén vào gốc repo.

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, urllib.request
hits = glob.glob('/kaggle/input/**/src/eval/judge.py', recursive=True)
assert hits, 'Không thấy src/eval/judge.py: kiểm tra đã Add Input dataset da2-judge chưa'
roots = [os.path.dirname(os.path.dirname(os.path.dirname(h))) for h in hits]
ok = [r for r in roots if os.path.exists(os.path.join(r, 'data/processed/judge_kappa/kappa_decisions.json'))]
assert ok, f'Không gốc nào có kappa_decisions.json: {roots}. Dataset da2-judge đang gắn là bản cũ?'
shutil.copytree(ok[0], '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')
print('Dataset root:', ok[0])
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print(subprocess.run('nvcc --version | tail -2', shell=True, capture_output=True, text=True).stdout)

## Bước 1. Dựng llama.cpp

`llama.cpp` là chương trình chạy LLM dạng file GGUF (trọng số đã lượng tử hóa 4-bit), có sẵn một server với API giống
OpenAI. Kaggle không có bản dựng sẵn cho GPU nên phải biên dịch từ mã nguồn, chỉ cho kiến trúc T4 (`sm_75`) để nhanh hơn.

Nếu đã lưu thư mục `/kaggle/working/llama_bin` của lượt trước thành một dataset và gắn vào notebook, ô này dùng lại bản
đó, khỏi biên dịch (và giữ đúng một phiên bản llama.cpp cho mọi lượt chấm). Phiên bản (commit) được ghi lại.

In [ ]:
BIN_DIR = '/kaggle/working/llama_bin'
os.makedirs(BIN_DIR, exist_ok=True)
pre = (glob.glob('/kaggle/input/**/llama_bin/llama-server', recursive=True)
       or glob.glob('/kaggle/input/**/llama-server', recursive=True))
use_pre = False
if pre:
    shutil.copy(pre[0], f'{BIN_DIR}/llama-server')
    os.chmod(f'{BIN_DIR}/llama-server', 0o755)
    v = os.path.join(os.path.dirname(pre[0]), 'VERSION.txt')
    if os.path.exists(v):
        shutil.copy(v, f'{BIN_DIR}/VERSION.txt')
    # Bản dựng sẵn có thể được biên dịch cho CPU khác (-march=native): chạy thử, hỏng thì biên dịch lại
    t = subprocess.run([f'{BIN_DIR}/llama-server', '--version'], capture_output=True, text=True)
    use_pre = t.returncode == 0
    print('Dùng llama-server dựng sẵn:' if use_pre else
          f'Bản dựng sẵn không chạy được trên máy này (mã {t.returncode}), biên dịch lại:', pre[0])
if not use_pre:
    t0 = time.time()
    src = '/tmp/llama.cpp'
    if not os.path.exists(src):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', src], check=True)
    commit = subprocess.run(['git', '-C', src, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
    shutil.rmtree(f'{src}/build', ignore_errors=True)   # xóa cấu hình CMake của lần chạy lỗi trước (nếu có)
    # GGML_CUDA_NO_VMM=ON: không liên kết trực tiếp với libcuda.so lúc biên dịch (Kaggle không có file này cho CMake
    # tìm, gây lỗi "CUDA::cuda_driver not found"). VMM chỉ là tối ưu cấp phát bộ nhớ, không đổi kết quả.
    cfg = subprocess.run(['cmake', '-S', src, '-B', f'{src}/build', '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75',
                          '-DGGML_CUDA_NO_VMM=ON', '-DGGML_NATIVE=OFF',   # chạy được trên mọi CPU Kaggle
                          '-DBUILD_SHARED_LIBS=OFF', '-DLLAMA_CURL=OFF',
                          '-DCMAKE_BUILD_TYPE=Release'],
                         capture_output=True, text=True)
    print(cfg.stdout[-1500:], cfg.stderr[-1500:])
    cfg.check_returncode()
    # Biên dịch mất 20–45 phút với 4 nhân CPU của Kaggle; in tiến độ (dòng "[ xx%]") mỗi khi tăng 5%,
    # giữ 60 dòng cuối để in khi lỗi.
    b = subprocess.Popen(['cmake', '--build', f'{src}/build', '--target', 'llama-server', '-j', str(os.cpu_count())],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail, last_pct = [], -5
    for line in b.stdout:
        tail = (tail + [line.rstrip()])[-60:]
        if line.startswith('[') and '%]' in line:
            pct = int(line[1:line.index('%')].strip() or 0)
            if pct >= last_pct + 5:
                last_pct = pct
                print(f'{pct:3d}% sau {(time.time() - t0) / 60:.1f} phút', flush=True)
    if b.wait() != 0:
        print('\n'.join(tail))
        raise RuntimeError('Biên dịch llama.cpp lỗi, xem các dòng cuối ở trên')
    shutil.copy(f'{src}/build/bin/llama-server', f'{BIN_DIR}/llama-server')
    open(f'{BIN_DIR}/VERSION.txt', 'w').write(commit + '\n')
    print(f'Dựng xong trong {(time.time() - t0) / 60:.1f} phút, commit {commit}')
os.chmod(f'{BIN_DIR}/llama-server', 0o755)
LLAMA_COMMIT = open(f'{BIN_DIR}/VERSION.txt').read().strip() if os.path.exists(f'{BIN_DIR}/VERSION.txt') else 'unknown'
print(subprocess.run([f'{BIN_DIR}/llama-server', '--version'], capture_output=True, text=True).stderr[-300:])
print('llama.cpp commit:', LLAMA_COMMIT)

## Bước 2. Cấu hình, tải model, các hàm bật/tắt server

- `CTX = 16384` token ngữ cảnh chia cho `PARALLEL = 8` luồng, mỗi luồng 2.048 token: đủ cho prompt hệ thống (~450 token)
  cộng câu hỏi, đáp án và câu trả lời dài nhất (tối đa 384 token).
- 8 luồng chấm song song (continuous batching): GPU xử lý nhiều câu một lúc nên nhanh hơn nhiều so với từng câu một.
- Model tải về `/tmp/models` (đĩa tạm, không tính vào 20 GB của `/kaggle/working`).

In [ ]:
from huggingface_hub import hf_hub_download
from src.eval import judge as J
from src.eval.score import contains_gold, exact_match

MODELS = {
    'qwen': dict(label='Qwen3.8-27B-UD-Q4_K_M', repo='unsloth/Qwen3.8-27B-GGUF', file='Qwen3.8-27B-UD-Q4_K_M.gguf',
                 gpus='0,1', extra=['--split-mode', 'layer', '--tensor-split', '1,1']),
    'gemma': dict(label='gemma-4-12b-it-qat-q4_0', repo='google/gemma-4-12B-it-qat-q4_0-gguf',
                  file='gemma-4-12b-it-qat-q4_0.gguf', gpus='0', extra=[]),
}
PRIMARY, CHECK = 'qwen', 'gemma'
SPEED_MAX = 2.0            # giây/câu; chậm hơn thì đổi vai (quy tắc khai báo trước)
CTX, PARALLEL, PORT = 16384, 8, 8080
os.environ['LLAMACPP_BASE_URL'] = f'http://127.0.0.1:{PORT}/v1'

def download(key):
    m = MODELS[key]
    t0 = time.time()
    m['path'] = hf_hub_download(m['repo'], m['file'], local_dir='/tmp/models')
    print(f"{m['file']}: {os.path.getsize(m['path']) / 1e9:.1f} GB, {time.time() - t0:.0f} s")

SERVER = {'proc': None}
def start_server(key):
    stop_server()
    m = MODELS[key]
    log = open(f"/kaggle/working/llama_{key}.log", 'w')
    cmd = [f'{BIN_DIR}/llama-server', '-m', m['path'], '-ngl', '999', '-c', str(CTX), '--parallel', str(PARALLEL),
           '--jinja', '--seed', '0', '--host', '127.0.0.1', '--port', str(PORT)] + m['extra']
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=m['gpus'])
    SERVER['proc'] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    t0 = time.time()
    while time.time() - t0 < 900:
        if SERVER['proc'].poll() is not None:
            raise RuntimeError(f"llama-server dừng, xem /kaggle/working/llama_{key}.log:\n" + open(log.name).read()[-3000:])
        try:
            if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5).status == 200:
                print(f"Server {m['label']} sẵn sàng sau {time.time() - t0:.0f} s")
                print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv'], capture_output=True, text=True).stdout)
                return
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError('llama-server không sẵn sàng sau 15 phút')

def stop_server():
    p = SERVER['proc']
    if p is not None and p.poll() is None:
        p.terminate()
        try:
            p.wait(60)
        except subprocess.TimeoutExpired:
            p.kill()
    SERVER['proc'] = None

for k in MODELS:
    download(k)

## Bước 3. Chấm 50 câu mẫu bằng từng judge, đo tốc độ và κ

50 câu `short` của VisR-Bench dev (seed cố định) đã được người duyệt (nhãn trong `kappa_decisions.json`). Mỗi judge chấm
50 câu này; đo số giây/câu (chỉ tính câu chấm mới, không tính câu lấy từ cache) và κ với nhãn người.

Mốc so sánh: luật đơn giản "câu trả lời chứa đáp án chuẩn" (không cần LLM). Judge LLM nên vượt mốc này; nếu không, nó
không đáng dùng.

In [ ]:
smp = json.load(open(J.KAPPA_DIR / 'kappa_sample.json', encoding='utf-8'))
items = smp['items']
human = {d['qid']: int(d['verdict'] == 'correct')
         for d in json.load(open(J.KAPPA_DIR / 'kappa_decisions.json', encoding='utf-8'))['decisions']}
qids = [it['qid'] for it in items]
h = [human[q] for q in qids]
rule = [int(any(contains_gold(it['pred'], g) for g in it['answers'])) for it in items]
print(f"Mốc 'chứa đáp án': κ = {J.cohen_kappa(h, rule):.3f}, khớp {sum(a == b for a, b in zip(h, rule))}/50")

STATS, VERDICTS = {}, {}
def judge_sample(key):
    m = MODELS[key]
    rows = [dict(it) for it in items]
    before = set(J.judge_rows([], 'llamacpp', m['label'], 1))          # khóa đã có trong cache
    t0 = time.time()
    cache = J.judge_rows(rows, 'llamacpp', m['label'], PARALLEL)
    sec = time.time() - t0
    keys = [J.item_key(m['label'], it['question'], it['answers'], it['pred']) for it in items]
    n_new = sum(k not in before for k in keys)
    judged = {q: {'judge': cache.get(k, (None, ''))[0], 'judge_reason': cache.get(k, (None, ''))[1],
                  'judge_model': m['label'], 'judge_version': J.JUDGE_VERSION} for q, k in zip(qids, keys)}
    res = J.compute_kappa(J.KAPPA_DIR / 'kappa_decisions.json', judged)
    VERDICTS[key] = [judged[q]['judge'] for q in qids]
    STATS[key] = {'label': m['label'], 'repo': m['repo'], 'file': m['file'], 'kappa': res['kappa'],
                  'agreement': res['agreement'], 'judge_correct': res['judge_correct'], 'human_correct': res['human_correct'],
                  'n_new': n_new, 'sec_total': round(sec, 1),
                  'sec_per_item': round(sec / n_new, 2) if n_new else None}
    print(STATS[key])

start_server(CHECK); judge_sample(CHECK)
start_server(PRIMARY); judge_sample(PRIMARY)

## Bước 4. Áp quy tắc chọn judge

Tốc độ đo ở bước 3. Nếu cả 50 câu của Qwen đã có trong cache (chạy lại notebook), dùng tốc độ đã ghi ở lượt trước
trong `results/judge/judge_selection.json`.

In [ ]:
sel_path = J.CACHE_DIR / 'judge_selection.json'
prev = json.load(open(sel_path, encoding='utf-8')) if sel_path.exists() else {}
for k in STATS:
    if STATS[k]['sec_per_item'] is None:
        STATS[k]['sec_per_item'] = prev.get('stats', {}).get(k, {}).get('sec_per_item')
        STATS[k]['speed_from_previous_run'] = True

q_speed = STATS[PRIMARY]['sec_per_item']
main = PRIMARY if (q_speed is not None and q_speed <= SPEED_MAX) else CHECK
reason = (f"Qwen {q_speed} s/câu <= {SPEED_MAX}" if main == PRIMARY else f"Qwen {q_speed} s/câu > {SPEED_MAX}: đổi vai")
if STATS[main]['kappa'] < J.KAPPA_MIN:
    reason += f"; nhưng κ judge được chọn = {STATS[main]['kappa']} < {J.KAPPA_MIN}: chỉ số chính quay về EM"
    main_ok = False
else:
    main_ok = True
selection = {
    'date': time.strftime('%Y-%m-%d'), 'judge_version': J.JUDGE_VERSION, 'llama_cpp_commit': LLAMA_COMMIT,
    'server': {'ctx': CTX, 'parallel': PARALLEL, 'seed': 0, 'temperature': 0},
    'rule': f'Qwen3.8-27B nếu <= {SPEED_MAX} s/câu, ngược lại Gemma 4 12B; judge được chọn cần κ >= {J.KAPPA_MIN}',
    'stats': STATS, 'kappa_between_judges': round(J.cohen_kappa(VERDICTS[PRIMARY], VERDICTS[CHECK]), 4),
    'kappa_rule_contains': round(J.cohen_kappa(h, rule), 4),
    'main_judge': MODELS[main]['label'] if main_ok else None, 'main_key': main if main_ok else None,
    'decision': reason,
}
json.dump(selection, open(sel_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(json.dumps(selection, ensure_ascii=False, indent=1))

## Bước 5. Judge được chọn chấm toàn bộ kết quả dev trong gói

Chấm mọi file `results/<nhánh>/dev_*.jsonl`. Mỗi file ra một file `*_judge.jsonl` cạnh nó (qid, đúng/sai, lý do).
Câu đã có trong cache (kể cả 50 câu mẫu) không chấm lại.

In [ ]:
if selection['main_key'] is None:
    print('Không có judge đạt κ: bỏ qua bước chấm toàn bộ, chỉ số chính quay về EM.')
else:
    if selection['main_key'] != PRIMARY:          # server đang chạy là Qwen; đổi sang Gemma
        start_server(selection['main_key'])
    label = MODELS[selection['main_key']]['label']
    summary = []
    for f in sorted(glob.glob('results/*/dev_*.jsonl')):
        if f.endswith('_judge.jsonl'):
            continue
        s = J.judge_file(f, 'llamacpp', label, PARALLEL)
        rows = [json.loads(l) for l in open(f, encoding='utf-8')]
        jd = {json.loads(l)['qid']: json.loads(l)['judge'] for l in open(f.replace('.jsonl', '_judge.jsonl'), encoding='utf-8')}
        for form in ['all', 'short']:
            sub = [r for r in rows if form == 'all' or r.get('answer_form') == form]
            if not sub:
                continue
            em = sum(max(exact_match(r['pred'], g) for g in (r.get('answers') or [r['gold']])) for r in sub) / len(sub)
            jj = sum(jd[r['qid']] == 1 for r in sub) / len(sub)
            summary.append((f, form, len(sub), round(100 * em, 1), round(100 * jj, 1)))
    for row in summary:
        print('%-45s %-5s n=%3d  EM %5.1f  judge %5.1f' % row)
stop_server()

## Bước 6. Đóng gói kết quả

`judge_results.zip` gồm `results/judge/` (cache, κ, quyết định chọn judge) và các file `*_judge.jsonl`. Tải về, giải
nén vào gốc repo.

Nên lưu thêm `/kaggle/working/llama_bin` thành một dataset (Output → New Dataset, tên ví dụ `da2-llama-bin`) để các
lượt chấm sau gắn vào, khỏi biên dịch lại và giữ đúng một phiên bản llama.cpp.

In [ ]:
out = '/kaggle/working/judge_results'
if os.path.exists(out):
    shutil.rmtree(out)
os.makedirs(f'{out}/results', exist_ok=True)
shutil.copytree('results/judge', f'{out}/results/judge')
for f in glob.glob('results/*/*_judge.jsonl'):
    os.makedirs(f'{out}/{os.path.dirname(f)}', exist_ok=True)
    shutil.copy(f, f'{out}/{f}')
shutil.make_archive(out, 'zip', out)
print('Tải /kaggle/working/judge_results.zip')